In [ ]:
import pandas as pd
import sqlite3

STEP 1: CREATE HISTORICAL E-COMMERCE DATA

In [ ]:
customers = pd.DataFrame({
    "Customer_ID": [1, 2, 3, 4, 5],
    "Customer_Name": ["Rahul", "Sneha", "Amit", "Priya", "Rohan"],
    "City": ["Mumbai", "Pune", "Delhi", "Mumbai", "Nashik"]
})

products = pd.DataFrame({
    "Product_ID": [101, 102, 103, 104],
    "Product_Name": ["Laptop", "Mobile", "Headphones", "Keyboard"],
    "Category": ["Electronics", "Electronics", "Accessories", "Accessories"],
    "Price": [50000, 25000, 3000, 1500]
})

orders = pd.DataFrame({
    "Order_ID": [1001, 1002, 1003, 1004, 1005],
    "Customer_ID": [1, 2, 3, 4, 5],
    "Product_ID": [101, 102, 103, 104, 101],
    "Quantity": [1, 2, 3, 2, 1],
    "Order_Date": [
        "2026-01-10",
        "2026-01-15",
        "2026-02-05",
        "2026-02-10",
        "2026-02-20"
    ]
})

payments = pd.DataFrame({
    "Payment_ID": [501, 502, 503, 504, 505],
    "Order_ID": [1001, 1002, 1003, 1004, 1005],
    "Payment_Mode": ["UPI", "Card", "UPI", "Cash", "Card"],
    "Payment_Status": ["Paid", "Paid", "Paid", "Paid", "Paid"]
})

# Save historical data as CSV files
customers.to_csv("/content/customers.csv", index=False)
products.to_csv("/content/products.csv", index=False)
orders.to_csv("/content/orders.csv", index=False)
payments.to_csv("/content/payments.csv", index=False)

print("Historical CSV files created successfully!")

Historical CSV files created successfully!


STEP 2: EXTRACT DATA FROM CSV USING PANDAS

In [ ]:
customers_df = pd.read_csv("/content/customers.csv")
products_df = pd.read_csv("/content/products.csv")
orders_df = pd.read_csv("/content/orders.csv")
payments_df = pd.read_csv("/content/payments.csv")

print("\n--- Customers ---")
print(customers_df)

print("\n--- Products ---")
print(products_df)

print("\n--- Orders ---")
print(orders_df)

print("\n--- Payments ---")
print(payments_df)


--- Customers ---
   Customer_ID Customer_Name    City
0            1         Rahul  Mumbai
1            2         Sneha    Pune
2            3          Amit   Delhi
3            4         Priya  Mumbai
4            5         Rohan  Nashik

--- Products ---
   Product_ID Product_Name     Category  Price
0         101       Laptop  Electronics  50000
1         102       Mobile  Electronics  25000
2         103   Headphones  Accessories   3000
3         104     Keyboard  Accessories   1500

--- Orders ---
   Order_ID  Customer_ID  Product_ID  Quantity  Order_Date
0      1001            1         101         1  2026-01-10
1      1002            2         102         2  2026-01-15
2      1003            3         103         3  2026-02-05
3      1004            4         104         2  2026-02-10
4      1005            5         101         1  2026-02-20

--- Payments ---
   Payment_ID  Order_ID Payment_Mode Payment_Status
0         501      1001          UPI           Paid
1         502 

STEP 3: DATA CLEANING

In [ ]:
# Remove duplicate records
customers_df = customers_df.drop_duplicates()
products_df = products_df.drop_duplicates()
orders_df = orders_df.drop_duplicates()
payments_df = payments_df.drop_duplicates()

# Handle missing values
customers_df["City"] = customers_df["City"].fillna("Unknown")
products_df["Category"] = products_df["Category"].fillna("Unknown")

# Convert date into proper date format
orders_df["Order_Date"] = pd.to_datetime(orders_df["Order_Date"])

# Remove invalid orders
orders_df = orders_df[
    (orders_df["Quantity"] > 0)
]

# Make payment status consistent
payments_df["Payment_Status"] = payments_df["Payment_Status"].str.title()

print("\nData cleaning completed successfully!")


Data cleaning completed successfully!


STEP 4: TRANSFORM DATA

In [ ]:
# Join orders with product information
sales_df = orders_df.merge(
    products_df,
    on="Product_ID",
    how="left"
)

# Calculate total amount
sales_df["Total_Amount"] = (
    sales_df["Quantity"] * sales_df["Price"]
)

print("\n--- Transformed Sales Data ---")
print(sales_df)


--- Transformed Sales Data ---
   Order_ID  Customer_ID  Product_ID  Quantity Order_Date Product_Name  \
0      1001            1         101         1 2026-01-10       Laptop   
1      1002            2         102         2 2026-01-15       Mobile   
2      1003            3         103         3 2026-02-05   Headphones   
3      1004            4         104         2 2026-02-10     Keyboard   
4      1005            5         101         1 2026-02-20       Laptop   

      Category  Price  Total_Amount  
0  Electronics  50000         50000  
1  Electronics  25000         50000  
2  Accessories   3000          9000  
3  Accessories   1500          3000  
4  Electronics  50000         50000  


STEP 5: CONNECT TO SQL DATABASE

In [ ]:
database = "/content/ecommerce.db"

connection = sqlite3.connect(database)

print("\nSQL database connected successfully!")


SQL database connected successfully!


STEP 6: LOAD DATA INTO SQL DATABASE

In [ ]:
customers_df.to_sql(
    "Customers",
    connection,
    if_exists="replace",
    index=False
)

products_df.to_sql(
    "Products",
    connection,
    if_exists="replace",
    index=False
)

orders_df.to_sql(
    "Orders",
    connection,
    if_exists="replace",
    index=False
)

payments_df.to_sql(
    "Payments",
    connection,
    if_exists="replace",
    index=False
)

sales_df.to_sql(
    "Sales_Report",
    connection,
    if_exists="replace",
    index=False
)

print("All data loaded into SQL database successfully!")

All data loaded into SQL database successfully!


STEP 7: GENERATE REPORT

In [ ]:
report_query = """
SELECT
    Category,
    SUM(Quantity) AS Total_Quantity,
    SUM(Total_Amount) AS Total_Sales
FROM Sales_Report
GROUP BY Category
ORDER BY Total_Sales DESC;
"""

report = pd.read_sql(report_query, connection)

print("\n--- SALES REPORT ---")
print(report)


--- SALES REPORT ---
      Category  Total_Quantity  Total_Sales
0  Electronics               4       150000
1  Accessories               5        12000


STEP 8: CUSTOMER SALES REPORT

In [ ]:
customer_query = """
SELECT
    c.Customer_Name,
    c.City,
    SUM(s.Total_Amount) AS Total_Spending
FROM Customers c
JOIN Orders o
    ON c.Customer_ID = o.Customer_ID
JOIN Sales_Report s
    ON o.Order_ID = s.Order_ID
GROUP BY c.Customer_ID
ORDER BY Total_Spending DESC;
"""

customer_report = pd.read_sql(
    customer_query,
    connection
)

print("\n--- CUSTOMER SALES REPORT ---")
print(customer_report)


--- CUSTOMER SALES REPORT ---
  Customer_Name    City  Total_Spending
0         Rohan  Nashik           50000
1         Sneha    Pune           50000
2         Rahul  Mumbai           50000
3          Amit   Delhi            9000
4         Priya  Mumbai            3000


STEP 9: HANDLE NEWLY ARRIVING DATA

In [ ]:
new_orders = pd.DataFrame({
    "Order_ID": [1006, 1007],
    "Customer_ID": [1, 3],
    "Product_ID": [102, 103],
    "Quantity": [1, 2],
    "Order_Date": ["2026-03-01", "2026-03-05"]
})

print("\n--- Newly Arriving Orders ---")
print(new_orders)



--- Newly Arriving Orders ---
   Order_ID  Customer_ID  Product_ID  Quantity  Order_Date
0      1006            1         102         1  2026-03-01
1      1007            3         103         2  2026-03-05


STEP 10: INCREMENTAL DATA LOADING

In [ ]:
# Get existing Order IDs from database
existing_orders = pd.read_sql(
    "SELECT Order_ID FROM Orders",
    connection
)

# Select only new orders
incremental_orders = new_orders[
    ~new_orders["Order_ID"].isin(
        existing_orders["Order_ID"]
    )
]

print("\n--- Records Selected for Incremental Loading ---")
print(incremental_orders)

# Load only new records
if not incremental_orders.empty:

    incremental_orders.to_sql(
        "Orders",
        connection,
        if_exists="append",
        index=False
    )

    print("\nIncremental loading completed successfully!")

else:
    print("\nNo new orders found.")


--- Records Selected for Incremental Loading ---
   Order_ID  Customer_ID  Product_ID  Quantity  Order_Date
0      1006            1         102         1  2026-03-01
1      1007            3         103         2  2026-03-05

Incremental loading completed successfully!


STEP 11: UPDATE SALES REPORT FOR NEW DATA

In [ ]:
new_sales = incremental_orders.merge(
    products_df,
    on="Product_ID",
    how="left"
)

new_sales["Total_Amount"] = (
    new_sales["Quantity"] * new_sales["Price"]
)

new_sales.to_sql(
    "New_Sales",
    connection,
    if_exists="replace",
    index=False
)

print("\n--- New Sales Data ---")
print(new_sales)


--- New Sales Data ---
   Order_ID  Customer_ID  Product_ID  Quantity  Order_Date Product_Name  \
0      1006            1         102         1  2026-03-01       Mobile   
1      1007            3         103         2  2026-03-05   Headphones   

      Category  Price  Total_Amount  
0  Electronics  25000         25000  
1  Accessories   3000          6000  


STEP 12: FINAL REPORT

In [ ]:
final_sales = pd.concat(
    [sales_df, new_sales],
    ignore_index=True
)

final_report = final_sales.groupby(
    "Category"
).agg(
    Total_Quantity=("Quantity", "sum"),
    Total_Sales=("Total_Amount", "sum")
).reset_index()

print("\n--- FINAL E-COMMERCE SALES REPORT ---")
print(final_report)



--- FINAL E-COMMERCE SALES REPORT ---
      Category  Total_Quantity  Total_Sales
0  Accessories               7        18000
1  Electronics               5       175000


STEP 13: VERIFY DATABASE

In [ ]:
print("\n--- Tables in SQL Database ---")

tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

print(tables)

connection.close()

print("\nETL PIPELINE COMPLETED SUCCESSFULLY!")


--- Tables in SQL Database ---
           name
0     Customers
1      Products
2        Orders
3      Payments
4  Sales_Report
5     New_Sales

ETL PIPELINE COMPLETED SUCCESSFULLY!
